In [1]:
# Cell 1: Import Thư viện
import sys
import os
import pandas as pd
import yfinance as yf
import joblib
import warnings

# Bỏ qua các cảnh báo không quan trọng
warnings.filterwarnings('ignore')
sys.path.append(os.path.abspath(".."))
from utils.feature_engineer import TechnicalFeatures

# Cell 2: Tải Mô hình AI đã huấn luyện
model_path = "../data/processed/xgboost_model.pkl"
model = joblib.load(model_path)

# Lấy danh sách chính xác các đặc trưng (features) mà mô hình yêu cầu
expected_features = model.feature_names_in_
print(f"Đã nạp Mô hình XGBoost. Mô hình yêu cầu {len(expected_features)} đặc trưng.")

# Cell 3: Tải dữ liệu thị trường mới nhất
ticker = "AAPL"  # Bạn có thể đổi thành "MSFT", "TSLA"... miễn là đã có model tương ứng
print(f"Đang tải dữ liệu thời gian thực cho {ticker}...")

# Ta cần tải ít nhất 100 ngày gần nhất để các chỉ báo (như SMA50) có đủ dữ liệu tính toán
df_live = yf.download(ticker, period="100d", interval="1d", progress=False)

# Xử lý lỗi định dạng MultiIndex của yfinance bản mới
if isinstance(df_live.columns, pd.MultiIndex):
    df_live.columns = df_live.columns.get_level_values(0)

# Cell 4: Xử lý dữ liệu (Feature Engineering)
te_live = TechnicalFeatures(df_live)
df_live_features = te_live.generate_all_features()

# Lấy dòng dữ liệu MỚI NHẤT (Phiên giao dịch gần nhất / Hôm nay)
today_data = df_live_features.iloc[[-1]].copy()
latest_date = today_data.index[0].strftime("%Y-%m-%d")
latest_close_price = today_data['Close'].values[0]

# Trích xuất đúng các cột mà mô hình XGBoost cần
X_live = today_data[expected_features]

# Cell 5: AI Đưa ra Dự đoán (Inference)
prediction = model.predict(X_live)[0]
probability = model.predict_proba(X_live)[0]

# In Báo cáo Tín hiệu
print("\n" + "★"*50)
print(f" TÍN HIỆU GIAO DỊCH AI - MÃ: {ticker} ")
print(f" Ngày cập nhật: {latest_date} | Giá đóng cửa: ${latest_close_price:.2f}")
print("★"*50)

if prediction == 1:
    print(">> KHUYẾN NGHỊ: MUA / NẮM GIỮ (BULLISH) 🟢")
    print(f">> Độ tự tin (Xác suất tăng): {probability[1]*100:.2f}%")
else:
    print(">> KHUYẾN NGHỊ: BÁN / ĐỨNG NGOÀI (BEARISH) 🔴")
    print(f">> Độ tự tin (Xác suất giảm): {probability[0]*100:.2f}%")
    
print("★"*50)

Đã nạp Mô hình XGBoost. Mô hình yêu cầu 10 đặc trưng.
Đang tải dữ liệu thời gian thực cho AAPL...


2026-07-30 13:55:12,800 [INFO] Bắt đầu tính toán Technical Indicators (sử dụng thư viện 'ta')...
2026-07-30 13:55:12,919 [INFO] Hoàn tất. Số lượng features: 19
2026-07-30 13:55:12,921 [INFO] Đã loại bỏ 49 dòng NaN ở đầu chuỗi dữ liệu.



★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
 TÍN HIỆU GIAO DỊCH AI - MÃ: AAPL 
 Ngày cập nhật: 2026-07-29 | Giá đóng cửa: $338.19
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
>> KHUYẾN NGHỊ: BÁN / ĐỨNG NGOÀI (BEARISH) 🔴
>> Độ tự tin (Xác suất giảm): 55.12%
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
